# ????RNN??,??GPU???????,???????????????

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gc
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
%matplotlib inline

from keras.models import Sequential, Model,load_model
from keras.layers import Bidirectional,Convolution1D,MaxPool1D,Flatten, TimeDistributed,RepeatVector
from keras.layers import SimpleRNN, GRU, LSTM, Conv1D, MaxPooling1D, GlobalAveragePooling1D, GlobalMaxPooling1D,SpatialDropout1D
from keras.layers import Dense, Activation, Reshape, Dropout, BatchNormalization, Input,concatenate
from keras.layers.embeddings import Embedding
from keras.optimizers import RMSprop, Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras import initializers
from keras import backend as K
from tensorflow import set_random_seed
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import MinMaxScaler

set_random_seed(15)

In [ ]:
train_data = pd.read_csv('input/train.csv')
test_data = pd.read_csv('input/test.csv')


## ?? 

In [ ]:
apis = list(set(list(train_data.api.unique())+list(test_data.api.unique())))
enc = LabelEncoder().fit(apis)
train_data['enc'] = enc.transform(train_data.api)
test_data['enc'] = enc.transform(test_data.api)


In [ ]:
tr = train_data.groupby('file_id').enc.apply(list).reset_index()
te = test_data.groupby('file_id').enc.apply(list).reset_index()


In [ ]:
label = train_data.groupby(['file_id'])['label'].agg('first').reset_index()

In [ ]:
from keras.utils.np_utils import to_categorical

categorical_labels = to_categorical(label.label, num_classes=None)


In [ ]:
df_train = pd.read_csv('feature/df_train_v4.csv')
df_test = pd.read_csv('feature/df_test_v4.csv')

df_train = pd.concat([df_train, pd.get_dummies(df_train.most_common_classes, prefix='most_common_classes')], axis=1)
df_test = pd.concat([df_test, pd.get_dummies(df_test.most_common_classes, prefix='most_common_classes')], axis=1)

df_train.drop('most_common_classes', axis=1, inplace=True)
df_test.drop('most_common_classes', axis=1, inplace=True)

cols = [c for c in df_train.columns if c not in ['file_id', 'label']]

df_train.fillna(0, inplace=True)
df_test.fillna(0, inplace=True)

scaler = MinMaxScaler().fit(df_train[cols])
mm_train = scaler.transform(df_train[cols])

mm_test = scaler.transform(df_test[cols])

In [ ]:
# ?????????????????5000?api
data = tr.enc.values

data = pad_sequences(data, 5000)

In [ ]:
## ?????
te_data = te.enc.values

te_data = pad_sequences(te_data, 7000)

## ????? 8-2?

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data, categorical_labels, test_size=0.2, random_state=42)

In [ ]:
num_train, num_val, y_train, y_val = train_test_split(mm_train, categorical_labels, test_size=0.2, random_state=42)

## RNN??

In [ ]:
def get_model():
    models = []
    inp = Input(shape=(5000, ))
    x = Embedding(max_enc+1, 50)(inp)
    x = Bidirectional(GRU(80?))(x)
    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    conc = concatenate([avg_pool, max_pool])
    models.append(conc)
    
    num_inp = Input(shape=(len(cols),) )
    x_num = Dense(256, activation="relu")(num_inp)
    x_num = Dense(64, activation="relu")(x_num)
    models.append(x_num)
    
    all_inp = concatenate(models)
    outp = Dense(6, activation="softmax")(all_inp)
    
    model = Model(inputs=[inp, num_inp], outputs=outp)
    model.compile(loss='categorical_crossentropy',
                  optimizer='adam')

    return model


In [ ]:
model = get_model()

model.fit([X_train, num_train], y_train,
          batch_size=256,
          epochs=20,
          validation_data=([X_val, num_val], y_val),
          verbose=1,
          callbacks=[
              EarlyStopping(monitor='val_loss', patience=1, verbose=1, mode='min'),
              ModelCheckpoint('/tmp/lstm_model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='min', period=1), 

              ]
         )

In [ ]:
model.load_weights('/tmp/lstm_model.h5')

In [ ]:
y_pred = model.predict([X_val, num_val])

In [ ]:
### ??loss

In [ ]:
log_loss(y_val, y_pred)

In [ ]:
y_sub = model.predict([te_data, mm_test])

## ??????

In [ ]:
sub = pd.read_csv('input/3rd_security_submit_sample.csv')

sub.loc[:, ['prob0','prob1','prob2','prob3','prob4','prob5']] = y_sub

sub = sub.round(7)

sub['sum'] = sub[['prob0','prob1','prob2','prob3','prob4','prob5']].sum(1)

sub['prob0'] = sub['prob0'] / sub['sum']
sub['prob1'] = sub['prob1'] / sub['sum']
sub['prob2'] = sub['prob2'] / sub['sum']
sub['prob3'] = sub['prob3'] / sub['sum']
sub['prob4'] = sub['prob4'] / sub['sum']
sub['prob5'] = sub['prob5'] / sub['sum']

sub.head()

sub[(sub[['prob0','prob1','prob2','prob3','prob4','prob5']].sum(1) - 1) >= 1e-6]

In [ ]:
sub[['file_id', 'prob0','prob1','prob2','prob3','prob4','prob5']].to_csv('rnn_02582.csv', index=False)

In [ ]:
np.save('rnn_y_pred_02582.npy', y_pred)